In [ ]:
import numpy as np
from orion.utils.o3d_utils import visualize_o3d_point_cloud, O3DPointCloud, estimate_rotation
from orion.utils.misc_utils import plotly_draw_3d_pcd, transform_points, create_point_clouds_from_keypoints
import open3d as o3d
from o3d_utils import draw_bbox_lines
import plotly.graph_objects as go
from matplotlib.colors import hsv_to_rgb
import os
import json
import h5py
import torch
from PIL import Image
import PIL

In [ ]:
def rainbow_colors(num_colors):
    colors = [(i/num_colors, 1, 1) for i in range(num_colors)]
    return 255 * np.array([hsv_to_rgb(color) for color in colors])

In [ ]:
out_dir = "/home/arisrei/research/RPL/ORION/visualization/centroid_trajs"
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/boat_gen6d_still"
demo_name = annotation_folder.split("/")[-1]
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)
img_indexes = np.array([0,30,60,90,126])
subsequence_range = [0, 25]

demo_file = h5py.File(config_dict["original_file"], "r")

depths = demo_file["data/human_demo/obs/dust3r_depth"][()][subsequence_range[0]:subsequence_range[1]]
images = demo_file["data/human_demo/obs/dust3r_rgb"][()][subsequence_range[0]:subsequence_range[1]]


intrinsics = demo_file["data/human_demo/obs/dust3r_intrinsics"][()][subsequence_range[0]:subsequence_range[1]]
#extrinsics = demo_file["data/human_demo/obs/dust3r_extrinsics"][()]
extrinsics = demo_file["data/human_demo/obs/dust3r_extrinsics_correct_frame"][()][subsequence_range[0]:subsequence_range[1]]
cotracker_dict = torch.load(os.path.join(annotation_folder, "tracked_points.pt"), map_location=torch.device('cuda:3'))
trajs_2d = cotracker_dict['pred_tracks'].cpu().detach().numpy()[0]
trajs_2d = trajs_2d[subsequence_range[0]:subsequence_range[1]]

scene_points = demo_file["data/human_demo/obs/scene_points"][()]
scene_colors = demo_file["data/human_demo/obs/scene_col"][()]

masks_orig = np.load(os.path.join(annotation_folder, 'masks.npz'))
masks_orig = masks_orig['arr_0']
masks_orig = masks_orig[subsequence_range[0]:subsequence_range[1]]

extrinsic_batches = demo_file["data/human_demo/obs/dust3r_extrinsic_batches"][()]
intrinsic_batches = demo_file["data/human_demo/obs/dust3r_intrinsic_batches"][()]
transforms_per_batch = demo_file['data/human_demo/obs/dust3r_transforms_per_batch'][()]
depths_per_batch = demo_file['data/human_demo/obs/dust3r_depths_unormalized_per_batch'][()]
demo_file.close()


In [ ]:
dust3r_mask = np.zeros((images.shape[0], images.shape[1], images.shape[2]), dtype=np.uint8)
for i in range(images.shape[0]):
    dust3r_mask[i] = np.array(Image.fromarray(masks_orig[i]).resize((images.shape[2], images.shape[1]), resample=PIL.Image.NEAREST))
display(Image.fromarray(dust3r_mask[0] * 100))

i = 0
target_ob_mask = 2
dust3r_mask[dust3r_mask != target_ob_mask] = 0 
dust3r_mask[dust3r_mask == target_ob_mask] = 1
masked = images[i] * dust3r_mask[i][:,:,np.newaxis]
display(Image.fromarray((images[i]).astype(np.uint8)))
display(Image.fromarray(masked.astype(np.uint8)))    
display(Image.fromarray(dust3r_mask[i]))

In [ ]:
initial_bbox = np.array([[13208.293, 8033.1484, 20.996002],    
                [13278.104,9047.406,  21.448826],    
                [14516.289,9347.258,  21.225342],    
                [14446.479,8332.999,  20.772518],    
                [14073.375,7733.4424, 22.037195],    
                [14143.186,8747.7,    22.490019],    
                [15381.371,9047.552,  22.266535],    
                [15311.561,8033.2935, 21.813711]])

blue = np.zeros_like(initial_bbox)
blue[:,2] = 255


In [ ]:
bbox_traj = np.load('/home/arisrei/research/RPL/ORION/third_party/Gen6D/data/custom/red_building-longer/test/images_out_smooth/3d-bbox-traj.npy')
poses = np.load('/home/arisrei/research/RPL/ORION/third_party/Gen6D/data/custom/red_building-longer/test/images_out_smooth/all_poses.npy')

#make poses 4x4 homogeneous
one = np.array([0,0,0,1])
one = one[None, None, :].repeat(poses.shape[0], axis=0)
poses = np.concatenate([poses, one], axis=1)

In [ ]:
#find transformation to make poses[0] identity
T = np.linalg.inv(poses[0])
poses_trf = np.matmul(T, poses)

In [ ]:

print(bbox_traj.shape)
tst = bbox_traj[1]
print(tst.shape)

red = np.zeros_like(tst)
colors = rainbow_colors(red.shape[0])
red = colors

pcd_wrapper = O3DPointCloud()
pcd_wrapper.pcd.points = o3d.utility.Vector3dVector(np.concatenate([tst]))
pcd_wrapper.pcd.colors = o3d.utility.Vector3dVector(np.concatenate([red]))

print(pcd_wrapper.get_points().shape)

fig = visualize_o3d_point_cloud(pcd_wrapper,marker_size=3, offline=True)

lines_trace = draw_bbox_lines(tst)
fig.add_trace(lines_trace)
#fig = go.Figure(data=[fig._data_objs[0], lines_trace], layout=fig.layout)

display(fig)

**Visualizes Boxes**

In [ ]:
colors = rainbow_colors(bbox_traj.shape[0])

bbox_traj_concat = np.concatenate(bbox_traj, axis=0)

#every 8 indices is a new bbox, change color
colors = np.repeat(colors, 8, axis=0)

print(colors.shape, bbox_traj_concat.shape)

pcd_wrapper = O3DPointCloud()
pcd_wrapper.pcd.points = o3d.utility.Vector3dVector(np.concatenate([bbox_traj_concat]))
pcd_wrapper.pcd.colors = o3d.utility.Vector3dVector(np.concatenate([colors]))

print(pcd_wrapper.get_points().shape)

#fig = visualize_o3d_point_cloud(pcd_wrapper,marker_size=3, offline=True)
equal_axis=True
fig = plotly_draw_3d_pcd(pcd_points=np.concatenate([bbox_traj_concat]), pcd_colors=np.concatenate([colors]), offline=True, equal_axis=equal_axis)


for i in range(bbox_traj.shape[0]):
    lines_trace = draw_bbox_lines(bbox_traj[i])
    fig.add_trace(lines_trace)

display(fig)
fig.write_html(os.path.join(out_dir, f"Gen6D_box_trajectories_eq_ax_{equal_axis}.html"))

**Visualize Centroids**

In [ ]:
#compute centroid of each bounding box
centroids = np.mean(bbox_traj, axis=1)
colors_centroids = rainbow_colors(centroids.shape[0])

pcd_wrapper = O3DPointCloud()
pcd_wrapper.pcd.points = o3d.utility.Vector3dVector(np.concatenate([centroids]))
pcd_wrapper.pcd.colors = o3d.utility.Vector3dVector(np.concatenate([colors_centroids]))

fig = plotly_draw_3d_pcd(addition_points=centroids, offline=True, additional_point_draw_lines=True, equal_axis=equal_axis)
display(fig)
fig.write_html(os.path.join(out_dir, f"Gen6D_centroid_trajectories_eq_ax_{equal_axis}.html"))

Apply 6DOF pose transformation matrices on some point in the point cloud to get the trajectory

In [ ]:
test_point = np.array([[0,0,0]])
test_point_traj = []
for i in range(poses_trf.shape[0]):
    test_point_traj.append(transform_points(poses_trf[i], test_point))
test_point_traj = np.concatenate(test_point_traj, axis=0)
print(test_point_traj.shape)

fig = plotly_draw_3d_pcd(addition_points=test_point_traj, offline=True, additional_point_draw_lines=True, equal_axis=equal_axis)
display(fig)


In [ ]:
keypoints = []
colors = []
point_to_trf_idx = None
for i in range(images[0].shape[0]):
    for j in range(images[0].shape[1]):
            keypoints.append(np.asarray([j, i]))
            colors.append(images[0][i, j])
            if type(point_to_trf_idx) != np.array and dust3r_mask[0][i, j] == 1:
                point_to_trf_idx = np.array([j, i])


keypoints = np.asarray(keypoints)
colors = np.asarray(colors)

points_3d = create_point_clouds_from_keypoints(keypoints, depths[0], intrinsics[0])
points_3d = points_3d * 1000

#remove points with depths less than trunc
trunc = 0.25
mask = points_3d[:,2] <= trunc
points_3d = points_3d[mask]
colors = colors[mask]


point_to_trf = create_point_clouds_from_keypoints(point_to_trf_idx[None, :], depths[0], intrinsics[0]) * 1000

In [ ]:
# print(points_3d[0])
# points_3d = points_3d - point_to_trf
# point_to_trf = point_to_trf - point_to_trf
# print(points_3d[0])
pcd_wrapper = O3DPointCloud()
pcd_wrapper.pcd = o3d.geometry.PointCloud()
pcd_wrapper.pcd.points = o3d.utility.Vector3dVector(points_3d)
pcd_wrapper.pcd.colors = o3d.utility.Vector3dVector(colors)
pcd_wrapper.max_points = 40000
pcd_wrapper.preprocess()



fig = plotly_draw_3d_pcd(pcd_wrapper.get_points(), pcd_wrapper.get_colors(), offline=True, additional_point_draw_lines=True, equal_axis=equal_axis, addition_points=point_to_trf)
display(fig)



In [ ]:
traj = []
for i in range(poses.shape[0]):
    traj.append(transform_points(poses[i], point_to_trf))
traj = np.concatenate(traj, axis=0)

pcd_wrapper.pcd.transform(poses[0])

new_origin = transform_points(poses[0], point_to_trf)
pcd_wrapper.pcd.points = o3d.utility.Vector3dVector(pcd_wrapper.get_points() - new_origin)
traj = traj - new_origin

traj = traj / 100

plane_estimation_kwargs={
        "ransac_n": 3,
        "num_iterations": 1000,
        "distance_threshold": 0.01
    }

plane_estimation_result = pcd_wrapper.plane_estimation(**plane_estimation_kwargs)
T_xy_plane_align = estimate_rotation(plane_estimation_result["plane_model"], z_up=False)
pcd_wrapper.pcd.transform(T_xy_plane_align)
traj = transform_points(T_xy_plane_align, traj)

#rotate traj by -90 deg about x-axis
T_x_rot = np.array([[1, 0, 0, 0],
                    [0, 0, 1, 0],
                    [0, -1, 0, 0],
                    [0, 0, 0, 1]])
traj = transform_points(T_x_rot, traj)


fig = plotly_draw_3d_pcd(pcd_wrapper.get_points(), pcd_wrapper.get_colors(), offline=True, additional_point_draw_lines=True, equal_axis=equal_axis, addition_points=traj)
display(fig)
fig.write_html(os.path.join(out_dir, f"dust3r_Gen6Dof_trajs.html"))